In [22]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input,Embedding,SimpleRNN,Dense

In [23]:
sentences = [
    # Positive (1)
    "I love learning machine learning.",
    "Artificial intelligence is amazing and exciting.",
    "She enjoys reading books every day.",
    "The weather is very pleasant today.",
    "He is happy with his new project.",
    "Data science is a fascinating field.",
    "They are having a great time in the park.",
    "I am enjoying Python programming.",
    "This is a beautiful place to visit.",
    "We are building an interesting neural network model.",
    "The cat looks very cute and calm.",
    "He likes to drink coffee in the morning.",
    "She is confident about her exams.",
    "The movie was very interesting and fun.",
    "I enjoy listening to good music.",

    # Negative (0)
    "I hate waiting in long lines.",
    "This task is very frustrating and tiring.",
    "She feels sad about the results.",
    "The weather is extremely bad today.",
    "He is disappointed with his performance.",
    "This is a boring and dull lecture.",
    "They are facing many problems in the project.",
    "I am not enjoying this work.",
    "This place looks dirty and unpleasant.",
    "We are struggling to complete the assignment.",
    "The cat looks sick and weak.",
    "He dislikes waking up early.",
    "She is worried about her exams.",
    "The movie was very boring.",
    "I am tired of doing the same thing."
]

labels = [1] * 15 + [0] * 15
labels = np.array(labels)

In [28]:
print(labels.size)
labels

30


array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0])

In [36]:
vocab_size = 2000
tokenizer = Tokenizer(num_words = vocab_size,oov_token = "<OOV>")
tokenizer.fit_on_texts(sentences)
sequence = tokenizer.texts_to_sequences(sentences)
maxlen = max(len(s) for s in sequence)
X = pad_sequences(sequence,maxlen = maxlen,padding="post")
Y = labels



In [39]:
X

array([[  5,  21,   6,  22,   6,   0,   0,   0],
       [ 23,  24,   3,  25,   2,  26,   0,   0],
       [  9,  27,  28,  29,  30,  31,   0,   0],
       [  2,  32,   3,  17,  33,  34,   0,   0],
       [ 10,   3,  18,  11,   4,  12,  19,   0],
       [ 35,  36,   3,  37,  38,  39,   0,   0],
       [ 13,   7,  40,  41,  14,   2,  42,   0],
       [  5,  15,   6,  43,  44,   0,   0,   0],
       [ 20,   3,   4,  45,  46,   8,  47,   0],
       [ 16,   7,  48,   4,  49,  50,  51,   0],
       [  2,  52,   3,  53,  11,   2,  54,   0],
       [ 10,  55,   8,  56,  57,  14,   2,  58],
       [  9,   3,  59,  60,  61,  62,   0,   0],
       [  2,  63,  64,  17,  65,   0,   0,   0],
       [  5,  66,  67,   8,  68,   0,   0,   0],
       [  2,  69,   3,  70,  71,   0,   0,   0],
       [ 13,  72,   8,   2,  73,  74,   0,   0],
       [ 16,   7,  75,  76,   6,  77,   0,   0],
       [  2,  78,  79,  80,  81,   0,   0,   0],
       [ 10,   3,  82,   4,  83,  84,   0,   0],
       [  9,   3,   

In [40]:
X.shape

(30, 8)

In [44]:
##Dimension of vector created by embedding layer will be of size 16
embed_dim = 16
## Number of neurons in hidden state
rnn_units = 8 
    

In [51]:
##Input layer what kind of data will enter my model
inp = Input(shape = (maxlen,),dtype = "int32",name = "input") 
x = Embedding(input_dim = vocab_size,output_dim = embed_dim,mask_zero = True,name = 'embed')(inp)
rnn = SimpleRNN(units = rnn_units,return_sequences = False,return_state = False,name="simple_rnn")
x_last = rnn(x)
out = Dense(1,activation = 'sigmoid',name = 'out')(x_last)
model = Model(inputs=inp,outputs=out)
model.compile(optimizer = 'adam',loss = 'binary_crossentropy',metrics = ['accuracy'])
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 8)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed (Embedding)   │ (None, 8, 16)     │     32,000 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_5         │ (None, 8)         │          0 │ input[0][0]       │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ (None, 8)         │        200 │ embed[0][0],      │
│ (SimpleRNN)         │                   │            │ not_equal_5[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ out (Dense)         │ (None, 1)         │          9 │ simple_rnn[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 32,209 (125.82 KB)

 Trainable params: 32,209 (125.82 KB)

 Non-trainable params: 0 (0.00 B)

In [53]:
model.fit(X,Y,epochs = 25,batch_size = 8,verbose = 1)

Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.3765 
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.3383 
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.3051 
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.2755 
Epoch 5/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.2479 
Epoch 6/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.2242 
Epoch 7/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.2030 
Epoch 8/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.1839 
Epoch 9/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.1678 
Epoch 10/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.1523 
Epoch 11/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.1401 
Epoch 12/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.1282 
E